In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available. Select T4 GPU from Runtime settings."
    )

print("GPU:", torch.cuda.get_device_name(0))

free, total = torch.cuda.mem_get_info()
print(f"Free memory: {free / 1024**3:.2f} GiB")
print(f"Total memory: {total / 1024**3:.2f} GiB")


In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

In [ ]:

import torch
import gc
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)

model.eval()

print("Model loaded successfully!")
print("GPU:", torch.cuda.get_device_name(0))


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!
GPU: Tesla T4


In [ ]:


prompt = "Explain what an operating system is in simple words."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        use_cache=True,
    )

answer = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(answer)

del inputs, output
gc.collect()
torch.cuda.empty_cache()




An operating system (OS) is a software that manages and controls the computer's hardware and software resources, allowing users to interact with the computer


In [ ]:


import time
import threading
import statistics
import torch
import gc

from transformers.generation.streamers import BaseStreamer


class TimingStreamer(BaseStreamer):
    def __init__(self):
        self.token_times = []
        self.prompt_received = False

    def put(self, value):
        # The first call contains the input prompt.
        if not self.prompt_received:
            self.prompt_received = True
            return

        torch.cuda.synchronize()
        self.token_times.append(time.perf_counter())

    def end(self):
        pass


def profile_generation(context_length, max_new_tokens=16):
    # Create an input with the requested number of tokens.
    seed_ids = tokenizer.encode(
        "Explain how a microcontroller works. ",
        add_special_tokens=False,
    )

    repeated_ids = (
        seed_ids * (context_length // len(seed_ids) + 1)
    )[:context_length]

    input_ids = torch.tensor(
        [repeated_ids],
        dtype=torch.long,
        device="cuda",
    )

    attention_mask = torch.ones_like(input_ids)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    baseline_memory = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()

    streamer = TimingStreamer()
    errors = []

    def generate():
        try:
            with torch.inference_mode():
                model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    streamer=streamer,
                    pad_token_id=tokenizer.eos_token_id,
                )
        except Exception as exc:
            errors.append(exc)

    torch.cuda.synchronize()
    start = time.perf_counter()

    worker = threading.Thread(target=generate)
    worker.start()
    worker.join()

    torch.cuda.synchronize()
    end = time.perf_counter()

    if errors:
        raise errors[0]

    times = streamer.token_times

    if not times:
        raise RuntimeError("No generated tokens were recorded.")

    ttft = times[0] - start

    intervals = [
        times[i] - times[i - 1]
        for i in range(1, len(times))
    ]

    mean_latency = (
        statistics.mean(intervals)
        if intervals else float("nan")
    )

    peak_memory = torch.cuda.max_memory_allocated()

    return {
        "context_tokens": context_length,
        "generated_tokens": len(times),
        "TTFT_seconds": ttft,
        "mean_inter_token_seconds": mean_latency,
        "total_generation_seconds": end - start,
        "baseline_allocated_GiB": baseline_memory / 1024**3,
        "peak_allocated_GiB": peak_memory / 1024**3,
    }


print("Profiler is ready!")


Profiler is ready!


In [ ]:


test_result = profile_generation(
    context_length=128,
    max_new_tokens=4,
)

print(test_result)

{'context_tokens': 128, 'generated_tokens': 4, 'TTFT_seconds': 0.4450609479999912, 'mean_inter_token_seconds': 0.06414102599993991, 'total_generation_seconds': 0.6379528129998562, 'baseline_allocated_GiB': 3.866405487060547, 'peak_allocated_GiB': 4.0145416259765625}


In [6]:
import pandas as pd
import gc
import torch

context_lengths = [1024, 2048, 4096, 8192]
results = []

for length in context_lengths:

    print(f"\nProfiling context length: {length}")

    for trial in range(1, 4):

        try:
            result = profile_generation(
                context_length=length,
                max_new_tokens=16,
            )

            results.append({
                "Context Length (Tokens)": length,
                "Trial": trial,
                "TTFT (Seconds)": result["TTFT_seconds"],
                "Inter-token Latency (Seconds)": result["mean_inter_token_seconds"],
                "VRAM Usage (GiB)": result["peak_allocated_GiB"],
            })

            print(f"Trial {trial} completed successfully.")

        except torch.cuda.OutOfMemoryError:

            print(f"Trial {trial}: OUT OF MEMORY")

            gc.collect()
            torch.cuda.empty_cache()

        except Exception as exc:

            print(f"Trial {trial}: ERROR - {exc}")

results_df = pd.DataFrame(results)

summary = (
    results_df.groupby("Context Length (Tokens)")
    .agg({
        "TTFT (Seconds)": "mean",
        "Inter-token Latency (Seconds)": "mean",
        "VRAM Usage (GiB)": "mean",
    })
    .reset_index()
)

display(summary.round(4))



Profiling context length: 1024
Trial 1 completed successfully.
Trial 2 completed successfully.
Trial 3 completed successfully.

Profiling context length: 2048
Trial 1 completed successfully.
Trial 2 completed successfully.
Trial 3 completed successfully.

Profiling context length: 4096
Trial 1 completed successfully.
Trial 2 completed successfully.
Trial 3 completed successfully.

Profiling context length: 8192
Trial 1: OUT OF MEMORY
Trial 2: OUT OF MEMORY
Trial 3: OUT OF MEMORY


,Context Length (Tokens),TTFT (Seconds),Inter-token Latency (Seconds),VRAM Usage (GiB)
0,1024,1.1439,0.1218,4.3866
1,2048,2.6397,0.1355,5.4692
2,4096,8.1626,0.1269,9.3453
